### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)
        * [Positive vs. Negative](#512-positive-vs-negative)




### 1. Environment Setup

##### 1.1 Library Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

PSD features are extracted from each 20s window using Welch's method. For each channel, log band power 
and relative band power are computed across the five standard frequency bands (delta, theta, alpha, beta, 
gamma), alongside time-domain mean and variance. Windows are then remapped into two binary classification schemes: Emotional vs. Neutral and Positive vs. Negative.

##### 3.1 PSD Feature Extraction

In [4]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {
        'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
    }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        # Band power per band
        ch_band_powers = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.trapz(Pxx[:, idx], f[idx], axis=1) 
            ch_band_powers.append(band_power)
        ch_band_powers = np.column_stack(ch_band_powers) 

        # Log band power
        log_band_power = np.log(ch_band_powers + 1e-10)
        all_features.append(log_band_power)

        # Relative band power
        total_power = ch_band_powers.sum(axis=1, keepdims=True)
        relative_band_power = ch_band_powers / (total_power + 1e-10)
        all_features.append(relative_band_power)

        # Time-domain features
        ch_mean = np.mean(ch_data, axis=1, keepdims=True)
        ch_var = np.var(ch_data, axis=1, keepdims=True)
        all_features.append(ch_mean)
        all_features.append(ch_var)

        
    
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [5]:
X, y = extract_psd_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 72)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (EN) ===
Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, cla

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


In [9]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (PN) ===
Subject 002: 167 windows, classes: [0 1], counts: [89 78]
Subject 003: 212 windows, classes: [0 1], counts: [ 65 147]
Subject 004: 141 windows, classes: [0 1], counts: [68 73]
Subject 005: 87 windows, classes: [0 1], counts: [33 54]
Subject 007: 433 windows, classes: [0 1], counts: [206 227]
Subject 012: 12 windows, classes: [0], counts: [12]
Subject 013: 14 windows, classes: [1], counts: [ 0 14]
Subject 015: 147 windows, classes: [0 1], counts: [ 22 125]
Subject 016: 29 windows, classes: [1], counts: [ 0 29]
Subject 017: 36 windows, classes: [0 1], counts: [28  8]
Subject 020: 10 windows, classes: [0], counts: [10]
Subject 021: 213 windows, classes: [0 1], counts: [ 28 185]
Subject 022: 95 windows, classes: [0 1], counts: [46 49]
Subject 023: 100 windows, classes: [0 1], counts: [21 79]
Subject 024: 235 windows, classes: [0 1], counts: [139  96]
Subject 025: 34 windows, classes: [1], counts: [ 0 34]
Subject 027: 26 windows, classes: [0 1], 

### 5. Model Training
For both models and classification schemes, hyperparameters are optimized using Optuna with 5-fold StratifiedGroupKFold on the full dataset. The resulting best parameters are fixed and used for final evaluation with LOSO. For comparison, performance is also assessed using standard 10-fold cross-validation.

##### General Functions for Training

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def xgb_hyperparameter_training(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [ ]:
def xgb_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #Per fold class weighting
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg/pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [ ]:
def xgb_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)
        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg / pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        # importance = model.feature_importances_ 
        # print(len(importance))

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [ ]:
#Hyperparameter tuning
xgb_en_params = xgb_hyperparameter_training(X_en, y_en, groups_en, "XGBoost", "Emotional vs. Neutral")

[I 2026-03-19 18:36:12,079] A new study created in memory with name: no-name-701354ba-c104-4c16-8f4b-4a96ae299cfd


=== Hyperparameter Tuning - XGBoost (Emotional vs. Neutral) ===


[I 2026-03-19 18:36:13,481] Trial 0 finished with value: 0.45899062138192903 and parameters: {'n_estimators': 134, 'max_depth': 3, 'learning_rate': 0.030827021715493917, 'subsample': 0.6258179645528632, 'colsample_bytree': 0.8811964025790557, 'min_child_weight': 9, 'gamma': 2.150536616821425}. Best is trial 0 with value: 0.45899062138192903.
[I 2026-03-19 18:36:21,570] Trial 1 finished with value: 0.4806623764099777 and parameters: {'n_estimators': 252, 'max_depth': 7, 'learning_rate': 0.13271975125761665, 'subsample': 0.7577230361393829, 'colsample_bytree': 0.6678202343978868, 'min_child_weight': 2, 'gamma': 0.9143364128776249}. Best is trial 1 with value: 0.4806623764099777.
[I 2026-03-19 18:36:23,912] Trial 2 finished with value: 0.49662234847279985 and parameters: {'n_estimators': 264, 'max_depth': 5, 'learning_rate': 0.2896235251596377, 'subsample': 0.771011602895594, 'colsample_bytree': 0.7623399852306668, 'min_child_weight': 6, 'gamma': 2.1739660752425154}. Best is trial 2 with 

Best params: {'n_estimators': 257, 'max_depth': 6, 'learning_rate': 0.26422889459941407, 'subsample': 0.7175249898229626, 'colsample_bytree': 0.7154789514203814, 'min_child_weight': 9, 'gamma': 0.043492737835852946}
Best CV F1: 0.4982


In [ ]:
#LOSO Evaluation
xgb_loso = xgb_loso_loop(X_en, y_en, groups_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== LOSO - XGBoost (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.5088 | Accuracy: 0.6054 | F1: 0.4515 | AUROC: 0.4744
Subject 003 | Balanced Accuracy: 0.4511 | Accuracy: 0.4450 | F1: 0.4419 | AUROC: 0.4408
Subject 004 | Balanced Accuracy: 0.4747 | Accuracy: 0.4694 | F1: 0.4687 | AUROC: 0.4571
Subject 005 | Balanced Accuracy: 0.5421 | Accuracy: 0.5620 | F1: 0.5390 | AUROC: 0.5400
Subject 007 | Balanced Accuracy: 0.5015 | Accuracy: 0.4579 | F1: 0.4097 | AUROC: 0.4952
Subject 012 | Balanced Accuracy: 0.6235 | Accuracy: 0.7158 | F1: 0.5644 | AUROC: 0.6898
Subject 013 | Balanced Accuracy: 0.4464 | Accuracy: 0.4231 | F1: 0.3780 | AUROC: 0.5119
Subject 015 | Balanced Accuracy: 0.4089 | Accuracy: 0.4126 | F1: 0.4090 | AUROC: 0.4088
Subject 016 | Balanced Accuracy: 0.5112 | Accuracy: 0.5822 | F1: 0.4640 | AUROC: 0.5127
Subject 017 | Balanced Accuracy: 0.3825 | Accuracy: 0.2864 | F1: 0.2780 | AUROC: 0.3172
Subject 020 | Balanced Accuracy: 0.5395 | Accuracy: 0.4095 | F1: 0.3608 

In [ ]:
#10Fold CV
xgb_10f = xgb_ten_fold_cv_loop(X_en, y_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== 10-Fold CV - XGBoost (Emotional vs. Neutral) ===
Accuracy:          0.6783 ± 0.0152
F1:                0.6759 ± 0.0146
Balanced Accuracy: 0.6758 ± 0.0144
AUROC:             0.7329 ± 0.0159


##### 5.1.2 Positive vs. Negative

In [ ]:
#Hyperparameter tuning
xgb_pn_params = xgb_hyperparameter_training(X_pn, y_pn, groups_pn, "XGBoost", "Positive vs. Negative")

[I 2026-03-19 18:39:44,049] A new study created in memory with name: no-name-74c1f9e1-3e8b-473c-bf18-70153fbeaa53


=== Hyperparameter Tuning - XGBoost (Positive vs. Negative) ===


[I 2026-03-19 18:39:46,293] Trial 0 finished with value: 0.5145818137474215 and parameters: {'n_estimators': 528, 'max_depth': 8, 'learning_rate': 0.29304137318491524, 'subsample': 0.9312672937010713, 'colsample_bytree': 0.6716906377115016, 'min_child_weight': 10, 'gamma': 2.2208917090662457}. Best is trial 0 with value: 0.5145818137474215.
[I 2026-03-19 18:39:48,535] Trial 1 finished with value: 0.5218673853028803 and parameters: {'n_estimators': 467, 'max_depth': 4, 'learning_rate': 0.12642526350548833, 'subsample': 0.8330838546101992, 'colsample_bytree': 0.839327847465113, 'min_child_weight': 4, 'gamma': 4.442879002660875}. Best is trial 1 with value: 0.5218673853028803.
[I 2026-03-19 18:39:52,761] Trial 2 finished with value: 0.5189976549005502 and parameters: {'n_estimators': 196, 'max_depth': 3, 'learning_rate': 0.050862880409523185, 'subsample': 0.6675966116706803, 'colsample_bytree': 0.7553708390818836, 'min_child_weight': 7, 'gamma': 1.3226224443604273}. Best is trial 1 with v

Best params: {'n_estimators': 243, 'max_depth': 7, 'learning_rate': 0.014569871423032144, 'subsample': 0.6090038757395747, 'colsample_bytree': 0.9146671105015334, 'min_child_weight': 10, 'gamma': 1.724260743988542}
Best CV F1: 0.5438


In [ ]:
#LOSO Evaluation
xgb_loso = xgb_loso_loop(X_pn, y_pn, groups_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== LOSO - XGBoost (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.5671 | Accuracy: 0.5808 | F1: 0.5540 | AUROC: 0.6570
Subject 003 | Balanced Accuracy: 0.4323 | Accuracy: 0.5519 | F1: 0.4203 | AUROC: 0.3440
Subject 004 | Balanced Accuracy: 0.6289 | Accuracy: 0.6241 | F1: 0.6192 | AUROC: 0.6612
Subject 005 | Balanced Accuracy: 0.5295 | Accuracy: 0.5402 | F1: 0.5261 | AUROC: 0.4652
Subject 007 | Balanced Accuracy: 0.5042 | Accuracy: 0.5012 | F1: 0.5004 | AUROC: 0.5051
Subject 015 | Balanced Accuracy: 0.5027 | Accuracy: 0.8231 | F1: 0.4868 | AUROC: 0.2182
Subject 017 | Balanced Accuracy: 0.7589 | Accuracy: 0.6944 | F1: 0.6630 | AUROC: 0.8080
Subject 021 | Balanced Accuracy: 0.5174 | Accuracy: 0.7934 | F1: 0.5181 | AUROC: 0.5747
Subject 022 | Balanced Accuracy: 0.5566 | Accuracy: 0.5474 | F1: 0.5107 | AUROC: 0.6557
Subject 023 | Balanced Accuracy: 0.4051 | Accuracy: 0.6400 | F1: 0.3902 | AUROC: 0.3828
Subject 024 | Balanced Accuracy: 0.3886 | Accuracy: 0.3872 | F1: 0.3847 

In [ ]:
#10Fold CV
xgb_10f = xgb_ten_fold_cv_loop(X_pn, y_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== 10-Fold CV - XGBoost (Positive vs. Negative) ===
Accuracy:          0.6720 ± 0.0261
F1:                0.6680 ± 0.0258
Balanced Accuracy: 0.6739 ± 0.0256
AUROC:             0.7422 ± 0.0267


#### KNN

In [54]:
def subject_normalize(X, groups):
    X_norm = X.copy()
    for subj in np.unique(groups):
        mask = groups == subj
        scaler = StandardScaler()
        X_norm[mask] = scaler.fit_transform(X[mask])
    return X_norm

In [55]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def knn_hyperparameter_training(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            groups_train = groups[train_idx]
            groups_val   = groups[val_idx]

            X_train_norm = subject_normalize(X_train, groups_train)
            X_val_norm   = subject_normalize(X_val, groups_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train_norm, y_train)
            preds = model.predict(X_val_norm)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    return best_params

In [56]:
def knn_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        groups_train = groups[train_idx]
        groups_test  = groups[test_idx]

        X_train_norm = subject_normalize(X_train, groups_train)
        X_test_norm  = subject_normalize(X_test, groups_test)

        model = KNeighborsClassifier(**params)
        model.fit(X_train_norm, y_train)
        preds = model.predict(X_test_norm)
        proba = model.predict_proba(X_test_norm)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [11]:
def knn_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)


        model = KNeighborsClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

In [58]:
#Hyperparameter tuning
knn_en_params = knn_hyperparameter_training(X_en, y_en, groups_en, "KNN", "Emotional vs. Neutral")

[I 2026-03-19 21:15:57,824] A new study created in memory with name: no-name-66b4d54b-5b73-46bd-b26b-c613074a59d1


=== Hyperparameter Tuning - KNN (Emotional vs. Neutral) ===


[I 2026-03-19 21:16:01,180] Trial 0 finished with value: 0.5019818328331377 and parameters: {'n_neighbors': 1, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 49}. Best is trial 0 with value: 0.5019818328331377.
[I 2026-03-19 21:16:04,509] Trial 1 finished with value: 0.5016399331709012 and parameters: {'n_neighbors': 23, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 49}. Best is trial 0 with value: 0.5019818328331377.
[I 2026-03-19 21:16:05,040] Trial 2 finished with value: 0.4979707904363929 and parameters: {'n_neighbors': 30, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 43}. Best is trial 0 with value: 0.5019818328331377.
[I 2026-03-19 21:16:05,257] Trial 3 finished with value: 0.4821465931988742 and parameters: {'n_neighbors': 22, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 16}. Best is trial 0 with value: 0.5019818328331377.
[I 2026-03-19 21:16:05,443] Trial 4 finished with value: 0.4971680010094216 and parameters: {'n_neighbors': 10, 'w

Best params: {'n_neighbors': 17, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 23}
Best CV F1: 0.5049


In [59]:
#LOSO Evaluation
knn_en_loso = knn_loso_loop(X_en, y_en, groups_en, knn_en_params, "KNN", "Emotional vs. Neutral")


=== LOSO - KNN (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.4413 | Accuracy: 0.5730 | F1: 0.4156 | AUROC: 0.4301
Subject 003 | Balanced Accuracy: 0.4754 | Accuracy: 0.4800 | F1: 0.4742 | AUROC: 0.4477
Subject 004 | Balanced Accuracy: 0.5008 | Accuracy: 0.4980 | F1: 0.4963 | AUROC: 0.5188
Subject 005 | Balanced Accuracy: 0.4980 | Accuracy: 0.4959 | F1: 0.4869 | AUROC: 0.5341
Subject 007 | Balanced Accuracy: 0.4912 | Accuracy: 0.5029 | F1: 0.4298 | AUROC: 0.4821
Subject 012 | Balanced Accuracy: 0.4252 | Accuracy: 0.4316 | F1: 0.3638 | AUROC: 0.3976
Subject 013 | Balanced Accuracy: 0.6190 | Accuracy: 0.6154 | F1: 0.6154 | AUROC: 0.6131
Subject 015 | Balanced Accuracy: 0.4974 | Accuracy: 0.5056 | F1: 0.4966 | AUROC: 0.4813
Subject 016 | Balanced Accuracy: 0.3635 | Accuracy: 0.4272 | F1: 0.3462 | AUROC: 0.3160
Subject 017 | Balanced Accuracy: 0.4442 | Accuracy: 0.3521 | F1: 0.3362 | AUROC: 0.4474
Subject 020 | Balanced Accuracy: 0.6184 | Accuracy: 0.3905 | F1: 0.3598 | AU

In [60]:
#10Fold CV
knn_en_10f = knn_ten_fold_cv_loop(X_en, y_en, knn_en_params, "KNN", "Emotional vs. Neutral")


=== 10-Fold CV - KNN (Emotional vs. Neutral) ===
Accuracy:          0.6249 ± 0.0137
F1:                0.6203 ± 0.0138
Balanced Accuracy: 0.6203 ± 0.0137
AUROC:             0.6726 ± 0.0163


In [61]:
#Hyperparameter tuning
knn_pn_params = knn_hyperparameter_training(X_pn, y_pn, groups_pn, "KNN", "Positive vs. Negative")

[I 2026-03-19 21:18:12,606] A new study created in memory with name: no-name-154d6afa-cfa3-4e9d-910c-e1545c81999d


=== Hyperparameter Tuning - KNN (Positive vs. Negative) ===


[I 2026-03-19 21:18:13,805] Trial 0 finished with value: 0.48810073682311916 and parameters: {'n_neighbors': 14, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 20}. Best is trial 0 with value: 0.48810073682311916.
[I 2026-03-19 21:18:14,088] Trial 1 finished with value: 0.4849202949915511 and parameters: {'n_neighbors': 23, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 39}. Best is trial 0 with value: 0.48810073682311916.
[I 2026-03-19 21:18:15,087] Trial 2 finished with value: 0.45872138297042603 and parameters: {'n_neighbors': 25, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 33}. Best is trial 0 with value: 0.48810073682311916.
[I 2026-03-19 21:18:15,261] Trial 3 finished with value: 0.4892593882921716 and parameters: {'n_neighbors': 2, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 30}. Best is trial 3 with value: 0.4892593882921716.
[I 2026-03-19 21:18:15,514] Trial 4 finished with value: 0.5118138598520814 and parameters: {'n_neighbors': 1

Best params: {'n_neighbors': 14, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 14}
Best CV F1: 0.5125


In [62]:
#LOSO Evaluation
knn_pn_loso = knn_loso_loop(X_pn, y_pn, groups_pn, knn_pn_params, "KNN", "Positive vs. Negative")


=== LOSO - KNN (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.6094 | Accuracy: 0.6048 | F1: 0.6044 | AUROC: 0.6490
Subject 003 | Balanced Accuracy: 0.5120 | Accuracy: 0.5613 | F1: 0.5093 | AUROC: 0.5241
Subject 004 | Balanced Accuracy: 0.5273 | Accuracy: 0.5319 | F1: 0.5213 | AUROC: 0.5131
Subject 005 | Balanced Accuracy: 0.5152 | Accuracy: 0.5517 | F1: 0.5148 | AUROC: 0.5466
Subject 007 | Balanced Accuracy: 0.4807 | Accuracy: 0.4873 | F1: 0.4739 | AUROC: 0.4752
Subject 015 | Balanced Accuracy: 0.5673 | Accuracy: 0.6463 | F1: 0.5218 | AUROC: 0.5535
Subject 017 | Balanced Accuracy: 0.6071 | Accuracy: 0.5278 | F1: 0.5092 | AUROC: 0.5960
Subject 021 | Balanced Accuracy: 0.5593 | Accuracy: 0.4977 | F1: 0.4368 | AUROC: 0.5863
Subject 022 | Balanced Accuracy: 0.4685 | Accuracy: 0.4737 | F1: 0.4563 | AUROC: 0.4907
Subject 023 | Balanced Accuracy: 0.4132 | Accuracy: 0.5700 | F1: 0.4188 | AUROC: 0.4491
Subject 024 | Balanced Accuracy: 0.5081 | Accuracy: 0.4638 | F1: 0.4517 | AU

In [63]:
#10Fold CV
#Most likely worse due to class imbalance
knn_pn_10f = knn_ten_fold_cv_loop(X_pn, y_pn, knn_pn_params, "KNN", "Positive vs. Negative")


=== 10-Fold CV - KNN (Positive vs. Negative) ===
Accuracy:          0.6417 ± 0.0312
F1:                0.6347 ± 0.0318
Balanced Accuracy: 0.6376 ± 0.0323
AUROC:             0.7069 ± 0.0288


##NEW

In [12]:
def knn_hyperparameter_training_10f(X, y, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning (10-Fold) - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")

    return study.best_params

In [13]:
knn_en_alt_params = knn_hyperparameter_training_10f(X_en, y_en, "knn", "Emotional vs. Neutral")

[I 2026-03-21 17:01:04,286] A new study created in memory with name: no-name-328e0a70-ca47-4e1d-88a0-8b107a4f2a51


=== Hyperparameter Tuning (10-Fold) - knn (Emotional vs. Neutral) ===


[I 2026-03-21 17:01:04,847] Trial 0 finished with value: 0.6202717356514501 and parameters: {'n_neighbors': 29, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 16}. Best is trial 0 with value: 0.6202717356514501.
[I 2026-03-21 17:01:05,274] Trial 1 finished with value: 0.6579800138205704 and parameters: {'n_neighbors': 16, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 27}. Best is trial 1 with value: 0.6579800138205704.
[I 2026-03-21 17:01:08,311] Trial 2 finished with value: 0.6171496227728132 and parameters: {'n_neighbors': 10, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 25}. Best is trial 1 with value: 0.6579800138205704.
[I 2026-03-21 17:01:08,431] Trial 3 finished with value: 0.6200502416433211 and parameters: {'n_neighbors': 17, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 32}. Best is trial 1 with value: 0.6579800138205704.
[I 2026-03-21 17:01:08,681] Trial 4 finished with value: 0.6132181041601447 and parameters: {'n_neighbors':

Best params: {'n_neighbors': 8, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 19}
Best CV F1: 0.6591


In [15]:
knn_pn_alt_params = knn_hyperparameter_training_10f(X_pn, y_pn, "knn", "Positive vs. Negative")

[I 2026-03-21 17:04:59,534] A new study created in memory with name: no-name-6d7887e8-c50e-41c1-9fc9-e57bd164c224
[I 2026-03-21 17:04:59,660] Trial 0 finished with value: 0.6048752767405946 and parameters: {'n_neighbors': 1, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 26}. Best is trial 0 with value: 0.6048752767405946.


=== Hyperparameter Tuning (10-Fold) - knn (Positive vs. Negative) ===


[I 2026-03-21 17:05:00,506] Trial 1 finished with value: 0.5797396062838402 and parameters: {'n_neighbors': 3, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 14}. Best is trial 0 with value: 0.6048752767405946.
[I 2026-03-21 17:05:01,487] Trial 2 finished with value: 0.5941981874549396 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 16}. Best is trial 0 with value: 0.6048752767405946.
[I 2026-03-21 17:05:01,631] Trial 3 finished with value: 0.6246673152857838 and parameters: {'n_neighbors': 4, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 45}. Best is trial 3 with value: 0.6246673152857838.
[I 2026-03-21 17:05:01,848] Trial 4 finished with value: 0.6232768232800115 and parameters: {'n_neighbors': 25, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 19}. Best is trial 3 with value: 0.6246673152857838.
[I 2026-03-21 17:05:02,741] Trial 5 finished with value: 0.5725216374451187 and parameters: {'n_neighbors': 17, 'w

Best params: {'n_neighbors': 24, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 33}
Best CV F1: 0.6364


In [16]:
knn_en_10f_alt = knn_ten_fold_cv_loop(X_en, y_en, knn_en_alt_params, "KNN", "Emotional vs. Neutral")


=== 10-Fold CV - KNN (Emotional vs. Neutral) ===
Accuracy:          0.6669 ± 0.0200
F1:                0.6626 ± 0.0213
Balanced Accuracy: 0.6623 ± 0.0211
AUROC:             0.7124 ± 0.0160


In [17]:
knn_pn_10f_alt = knn_ten_fold_cv_loop(X_pn, y_pn, knn_pn_alt_params, "KNN", "Positive vs. Negative")


=== 10-Fold CV - KNN (Positive vs. Negative) ===
Accuracy:          0.6447 ± 0.0269
F1:                0.6343 ± 0.0261
Balanced Accuracy: 0.6351 ± 0.0254
AUROC:             0.7058 ± 0.0296
